# Examples

Short, copy-pasteable recipes for common tasks. Each is self-contained given the
setup cell below. For the concepts behind them, see the
[User Guide](userguide.ipynb).

In [1]:
import os
import pandas as pd
from euromod import Model

# --- Point these to your local EUROMOD installation -------------------------
# MODEL_PATH : a EUROMOD model/release folder (the one that contains 'XMLParam').
# DATA_DIR   : folder with the input microdata (.txt). In a public release this
#              is the model's own 'Input' folder, which ships with the freely
#              distributable *training* datasets used throughout these docs.
MODEL_PATH = r"C:\EUROMOD\EUROMOD_RELEASE"
DATA_DIR   = os.path.join(MODEL_PATH, "Input")

# keep DataFrame previews compact in the rendered docs
pd.set_option("display.max_rows", 8, "display.max_columns", 8)

Using EUROMOD as defined in C:\EUROMOD\Executable


In [3]:
mod = Model(MODEL_PATH)

## Run a baseline simulation

In [4]:
data = pd.read_csv(os.path.join(DATA_DIR, "AT_training_data.txt"), sep="\t")
sim = mod.countries["AT"].systems["AT_2025"].run(data, "AT_training_data", verbose=False)
sim.outputs[0].ils_dispy.mean()

1694.418331768049

## Compare a reform against the baseline

Switch a policy off and measure the effect on mean disposable income.

In [5]:
data = pd.read_csv(os.path.join(DATA_DIR, "IT_training_data.txt"), sep="\t")
sys = mod.countries["IT"].systems["IT_2020"]

base = sys.run(data, "IT_training_data", verbose=False)
bfacc = [p for p in sys.policies if p.name == "bfacc_it"][0]   # policies are keyed by ID
bfacc.switch = "off"
reform = sys.run(data, "IT_training_data", verbose=False)
bfacc.switch = "on"                             # reset

reform.outputs[0].ils_dispy.mean() - base.outputs[0].ils_dispy.mean()

-0.2886928628709029

## Run with an add-on (name only)

Pass the add-on by name and let EUROMOD resolve the applicable add-on system
automatically.

In [6]:
sim = mod.countries["IT"].systems["IT_2020"].run(
    data, "IT_training_data", addons=["MTR"], verbose=False)
sim.output_filenames

['it_2020_base_mtr.txt', 'it_2020_mtr.txt']

Read a value from the add-on's own output:

In [7]:
sim.outputs["it_2020_mtr.txt"].mtrpc.mean()

20.76796201407752

## Switch on an extension

In [8]:
sim = mod.countries["IT"].systems["IT_2020"].run(
    data, "IT_training_data", switches=[("BTA", True)], verbose=False)
sim.outputs[0].ils_ben.mean()

435.3544787117912

## Overwrite a constant

In [9]:
sim = mod.countries["IT"].systems["IT_2020"].run(
    data, "IT_training_data",
    constantsToOverwrite={("$penIndexA_2007", ""): "1.05"},
    verbose=False)
sim.constantsToOverwrite

{('$penIndexA_2007', ''): '1.05'}

## Request only specific output variables

In [10]:
sim = mod.countries["IT"].systems["IT_2020"].run(
    data, "IT_training_data",
    requested_vars=["ils_dispy", "ils_origy"],
    verbose=False)
sim.outputs["custom_output.txt"].head()

,ils_dispy,ils_origy,idperson
0,780.000000,0.00000,101.0
1,780.000000,0.00000,201.0
2,780.000000,123.66000,301.0
3,780.000000,494.63980,401.0
4,844.142738,865.61963,501.0


## Compare several countries

In [11]:
rows = []
for cc, sysname in [("AT", "AT_2025"), ("IT", "IT_2020"), ("SL", "SL_1996")]:
    d = pd.read_csv(os.path.join(DATA_DIR, f"{cc}_training_data.txt"), sep="\t")
    s = mod.countries[cc].systems[sysname].run(d, f"{cc}_training_data", verbose=False)
    rows.append((cc, sysname, s.outputs[0].ils_dispy.mean()))

pd.DataFrame(rows, columns=["country", "system", "mean_disposable_income"])

,country,system,mean_disposable_income
0,AT,AT_2025,1694.418332
1,IT,IT_2020,1084.507587
2,SL,SL_1996,1061.144695
